# Group 2: Stock Factor Betas (Steps 1-5)

Estimate each stock's currency, AUD, market, and sector betas on stagger 2 by regressing
on one factor at a time and carrying the residuals forward. All data comes from group0's
`companies_and_staggered_returns.xlsx`.

**Currency rule:** a stock only ever gets a beta on its own home currency (kept as a prior)
and on AUD (the one other currency we test). Its betas on MXN, SGD, CAD, DKK and EUR are 0
unless that currency is its home currency. US stocks have no home currency, so AUD is the
only currency they are tested on.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm

DATA_FILE = Path("../data/companies_and_staggered_returns.xlsx")
STAGGER = "stagger_2"


## Load the data

Get the stock list from the `Companies` sheet, and load stagger 2's time-weighted
returns and residual sector factors.

In [2]:
companies = pd.read_excel(DATA_FILE, sheet_name="Companies")

# Keep only real stocks: one row per ticker, and no ETF / FX / SPY rows
stocks = (
    companies
    .dropna(subset=["sector"])
    .drop_duplicates(subset=["ticker"])
    .reset_index(drop=True)
)

print(f"{len(stocks)} stocks")
print(f"Foreign stocks: {stocks.loc[stocks['currency'] != 'USD', 'ticker'].tolist()}")
stocks

43 stocks
Foreign stocks: ['VIST', 'STX', 'NTR', 'ASND', 'NOK']


,ticker,company_name,country,sector,currency
0,VST,Vistra Corp.,United States,Utilities,USD
1,VIST,"Vista Energy, S.A.B. de C.V.",Mexico,Energy,MXN
2,AEP,"American Electric Power Company, Inc.",United States,Utilities,USD
3,ETR,Entergy Corporation,United States,Utilities,USD
4,DIS,The Walt Disney Company,United States,Communication Services,USD
5,CMG,"Chipotle Mexican Grill, Inc.",United States,Consumer Discretionary,USD
6,AMZN,"Amazon.com, Inc.",United States,Consumer Discretionary,USD
7,FIX,"Comfort Systems USA, Inc.",United States,Industrials,USD
8,STX,Seagate Technology Holdings plc,Singapore,Technology,SGD
9,VLO,Valero Energy Corporation,United States,Energy,USD


In [3]:
returns = pd.read_excel(
    DATA_FILE, sheet_name=STAGGER, skiprows=254, nrows=80, parse_dates=["date"]
).set_index("date")
sector_factors = pd.read_excel(
    DATA_FILE, sheet_name=STAGGER, skiprows=338, nrows=80, parse_dates=["date"]
).set_index("date")

print(returns.shape)
returns.head()

(80, 61)


,VST,VIST,AEP,ETR,DIS,CMG,AMZN,FIX,STX,VLO,...,XLB,XLRE,XLK,XLU,MXN/USD,SGD/USD,CAD/USD,DKK/USD,EUR/USD,AUD/USD
date,,,,,,,,,,,,,,,,,,,,,
2020-07-27,-0.010961,0.063846,0.027994,0.044546,0.022594,0.048566,0.049309,0.118025,-0.024910,-0.001063,...,0.026109,0.009377,0.026018,0.026840,0.002653,0.007938,0.000174,0.025351,0.025030,0.021907
2020-08-24,0.018595,-0.089985,-0.052445,-0.028306,0.081393,0.065777,0.039842,0.017341,0.016576,-0.003605,...,0.032031,0.004694,0.060161,-0.016190,0.013116,0.007092,0.013256,0.006324,0.005959,0.005046
2020-09-21,-0.034672,-0.174135,0.016181,-0.011303,-0.049765,-0.025050,-0.052835,-0.008918,0.026700,-0.124691,...,-0.010732,-0.026482,-0.042279,0.001466,-0.015061,-0.007518,-0.012321,-0.013217,-0.013067,-0.016609
2020-10-19,0.038649,0.002584,0.073941,0.066012,0.019556,0.041546,0.019679,0.050786,0.028893,-0.025759,...,0.031268,0.015492,0.021970,0.055445,0.038954,0.007224,0.011094,0.011394,0.011153,0.005514
2020-11-16,-0.033161,0.059561,-0.057314,0.005187,0.054358,-0.020335,-0.019165,-0.063692,0.052720,0.120281,...,0.031872,0.020770,0.008054,-0.010852,0.021316,0.006049,0.001317,-0.000735,-0.000345,0.011429


In [4]:
# Map each GICS sector to its SPDR sector ETF (the residual factor columns
# in group0's file are named like 'XLB_residual')
sector_to_etf = {
    "Communication Services": "XLC",
    "Consumer Discretionary": "XLY",
    "Consumer Staples": "XLP",
    "Energy": "XLE",
    "Financials": "XLF",
    "Health Care": "XLV",
    "Industrials": "XLI",
    "Materials": "XLB",
    "Real Estate": "XLRE",
    "Technology": "XLK",
    "Utilities": "XLU",
}

# The FX return columns are named like 'MXN/USD', so a stock's currency code
# tells us which column to use
def fx_column(currency):
    return currency + "/USD"


## The regression helpers

Every step is an OLS regression with an intercept; the residual carried to the next step
is `return - (alpha + beta * factor)`, same as group0.

`regress` is the single-factor regression used for the priors (home currency, SPY, own
sector). `regress_stepwise` is used for the factors that have to earn their place: it
regresses on all candidates together, drops the one whose t-stat is closest to zero if that
t-stat is between -2 and 2, and refits, until every remaining factor has |t| >= 2 or none
are left.

In [ ]:
T_CUTOFF = 2.0


def regress(y, x):
    """Regress y on x (with an intercept). Return beta, alpha, and beta's t-stat."""
    X = sm.add_constant(x)
    model = sm.OLS(y, X, missing="drop").fit()
    beta = model.params.iloc[1]
    alpha = model.params.iloc[0]
    t_stat = model.tvalues.iloc[1]
    return beta, alpha, t_stat


def regress_stepwise(y, X):
    """Backward stepwise regression of y on the columns of X (with an intercept).

    Each round, drop the factor whose t-stat is closest to zero if that t-stat is
    between -2 and 2, then refit. Stop when every remaining factor has |t| >= 2 or
    none are left.

    Returns (betas, t_stats, fitted): dicts of the kept factors' betas and t-stats,
    and the fitted part (alpha + X @ beta) to subtract, or None if nothing was kept.
    """
    kept = list(X.columns)
    while kept:
        model = sm.OLS(y, sm.add_constant(X[kept]), missing="drop").fit()
        t = model.tvalues[kept]
        weakest = t.abs().idxmin()
        if abs(t[weakest]) < T_CUTOFF:
            kept.remove(weakest)
            continue
        betas = model.params[kept]
        fitted = model.params["const"] + X[kept] @ betas
        return betas.to_dict(), t.to_dict(), fitted
    return {}, {}, None

## Steps 1-5: the stepwise regressions

For each stock:

1. **Home currency** (foreign stocks only): regress on the stock's own currency and keep
   the beta. US stocks skip this step.
2. **AUD**: regress the residual on AUD and keep the beta only if |t| >= 2, otherwise set
   it to 0. AUD is the only non-home currency tested, so every other currency beta is 0.
3. **Market**: regress the residual on SPY and keep the beta.
4. **Own sector**: regress the residual on the stock's own residual sector factor and keep
   the beta.
5. **Other sectors**: regress the residual on the other 10 sector factors together and
   drop them stepwise (same |t| < 2 rule as above). Dropped sectors get a beta of 0.

What's left is the stock-specific return.

In [ ]:
CURRENCIES = ["MXN", "SGD", "CAD", "DKK", "EUR", "AUD"]
TESTED_CURRENCY = "AUD"          # the only non-home currency we regress on
SECTOR_ETFS = sorted(sector_to_etf.values())

beta_rows = []
aud_t_stats = {}
stock_specific = pd.DataFrame(index=returns.index)

for _, stock in stocks.iterrows():
    ticker = stock["ticker"]
    home = stock["currency"]
    r = returns[ticker].astype(float)

    # Every currency beta starts at 0; only the home currency and AUD can change it
    currency_betas = dict.fromkeys(CURRENCIES, 0.0)

    # Step 1: own home currency (prior, keep the beta). US stocks skip this.
    if home != "USD":
        fx = returns[fx_column(home)].astype(float)
        beta_fx, alpha, _ = regress(r, fx)
        r = r - (alpha + beta_fx * fx)
        currency_betas[home] = beta_fx

    # Step 2: AUD only (not a prior, must pass |t| >= 2). Skipped if AUD is the home currency.
    if home != TESTED_CURRENCY:
        aud = returns[[fx_column(TESTED_CURRENCY)]].astype(float)
        kept, t_stats, fitted = regress_stepwise(r, aud)
        aud_t_stats[ticker] = regress(r, aud.iloc[:, 0])[2]
        if fitted is not None:
            r = r - fitted
            currency_betas[TESTED_CURRENCY] = kept[fx_column(TESTED_CURRENCY)]

    # Step 3: market (prior, keep the beta)
    spy = returns["SPY"].astype(float)
    beta_mkt, alpha, _ = regress(r, spy)
    r = r - (alpha + beta_mkt * spy)

    # Step 4: own residual sector factor (prior, keep the beta)
    own_etf = sector_to_etf[stock["sector"]]
    sector = sector_factors[own_etf + "_residual"].astype(float)
    beta_sector, alpha, _ = regress(r, sector)
    r = r - (alpha + beta_sector * sector)

    # Step 5: the other 10 sector factors, dropped stepwise
    sector_betas = dict.fromkeys(SECTOR_ETFS, 0.0)
    sector_betas[own_etf] = beta_sector
    others = [etf for etf in SECTOR_ETFS if etf != own_etf]
    X_other = sector_factors[[etf + "_residual" for etf in others]].astype(float)
    kept, _, fitted = regress_stepwise(r, X_other)
    if fitted is not None:
        r = r - fitted
        for col, beta in kept.items():
            sector_betas[col.replace("_residual", "")] = beta

    # r is now the stock-specific return
    stock_specific[ticker] = r
    beta_rows.append({
        "ticker": ticker,
        "sector": stock["sector"],
        "currency": home,
        **{("cur", c): b for c, b in currency_betas.items()},
        "SPY": beta_mkt,
        **{("sec", e): b for e, b in sector_betas.items()},
    })

# Two-level column header: Stock | Currency betas | Market | Sector betas
column_index = pd.MultiIndex.from_tuples(
    [("Stock", "ticker"), ("Stock", "sector"), ("Stock", "currency")]
    + [("Currency betas", c) for c in CURRENCIES]
    + [("Market", "SPY")]
    + [("Sector betas", e) for e in SECTOR_ETFS]
)
betas = pd.DataFrame(
    [[row["ticker"], row["sector"], row["currency"]]
     + [row[("cur", c)] for c in CURRENCIES]
     + [row["SPY"]]
     + [row[("sec", e)] for e in SECTOR_ETFS]
     for row in beta_rows],
    columns=column_index,
)

cur = betas["Currency betas"]
print(f"{len(betas)} stocks on {STAGGER}")
print(f"AUD beta kept (|t| >= 2): {(cur['AUD'] != 0).sum()} of {len(betas)}")
print(f"Other sector betas kept: {(betas['Sector betas'] != 0).sum().sum() - len(betas)}")
betas.round(3)

## Verification 1: the currency rule and orthogonality

First check that no stock has a beta on a currency that is neither its home currency nor
AUD. Then check how correlated the stock-specific returns still are with the factors that
were removed. These are close to 0 but not exactly 0, because each later step (AUD, SPY,
the other sectors) subtracts something that co-moves a little with the earlier factors.
The foreign stocks' local-currency correlation is the largest of these.

In [ ]:
# 1) Currency rule: only the home currency and AUD may be non-zero
stray = [
    (row[("Stock", "ticker")], c)
    for _, row in betas.iterrows()
    for c in CURRENCIES
    if c not in (row[("Stock", "currency")], TESTED_CURRENCY)
    and row[("Currency betas", c)] != 0
]
assert not stray, f"currency betas that should be 0: {stray}"
print("Currency rule holds: every non-home, non-AUD currency beta is 0")

print("AUD t-stats (|t| >= 2 kept):")
print(pd.Series(aud_t_stats).round(2).to_string())

# 2) Orthogonality
checks = []
for _, stock in stocks.iterrows():
    ticker = stock["ticker"]
    r = stock_specific[ticker]
    etf = sector_to_etf[stock["sector"]]
    row = {
        "ticker": ticker,
        "corr_spy": r.corr(returns["SPY"]),
        "corr_sector": r.corr(sector_factors[etf + "_residual"]),
    }
    # only meaningful where the factor was actually removed
    if stock["currency"] != "USD":
        row["corr_fx"] = r.corr(returns[fx_column(stock["currency"])])
    checks.append(row)

checks = pd.DataFrame(checks)
print()
print("Largest correlation (absolute value) between the stock-specific returns")
print("and each removed factor:")
checks[["corr_spy", "corr_sector", "corr_fx"]].abs().max().round(6)

## Verification 2: a visual example

One stock's returns vs SPY before the regressions, and its stock-specific returns after
(no market relationship left).

In [ ]:
import matplotlib.pyplot as plt

EXAMPLE = "AMZN"

raw = returns[EXAMPLE].astype(float)
spy = returns["SPY"].astype(float)
final = stock_specific[EXAMPLE]

beta_raw, alpha_raw, _ = regress(raw, spy)
line_x = np.linspace(spy.min(), spy.max(), 100)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].scatter(spy, raw, alpha=0.7)
axes[0].plot(line_x, alpha_raw + beta_raw * line_x, color="red")
axes[0].set_title(f"{EXAMPLE} returns vs SPY (beta = {beta_raw:.2f})")
axes[0].set_xlabel("SPY time-weighted return")
axes[0].set_ylabel(f"{EXAMPLE} time-weighted return")
axes[0].grid(alpha=0.2)

axes[1].scatter(spy, final, alpha=0.7, color="green")
axes[1].axhline(0, color="red")
axes[1].set_title(f"{EXAMPLE} stock-specific returns vs SPY (no relationship)")
axes[1].set_xlabel("SPY time-weighted return")
axes[1].set_ylabel("stock-specific return")
axes[1].grid(alpha=0.2)

plt.tight_layout()
plt.show()

## Save the results

Write the stagger 2 betas, the stock-specific returns, and the 43 x 43 covariance matrix
of those returns to one Excel file. A dropped factor is stored as a beta of 0. The
covariance uses pairwise-complete observations (DOCN and UPST start later than the rest),
so the matrix is symmetric.

In [ ]:
from openpyxl.styles import Alignment, Font

output_file = Path("group2_stock_betas.xlsx")

covariance = stock_specific.cov()
assert np.allclose(covariance, covariance.T), "covariance matrix is not symmetric"

residuals_out = stock_specific.copy()
residuals_out.index = residuals_out.index.strftime("%Y-%m-%d")

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    # Betas: write the two header rows by hand so there is no index column
    (betas.set_axis(range(betas.shape[1]), axis=1)
     .to_excel(writer, sheet_name="Betas", startrow=2, header=False, index=False))
    ws = writer.sheets["Betas"]
    bold = Font(bold=True)
    groups = betas.columns.get_level_values(0)
    for j, (top, sub) in enumerate(betas.columns, start=1):
        ws.cell(row=2, column=j, value=sub).font = bold
        if j == 1 or groups[j - 2] != top:
            start = j
            ws.cell(row=1, column=j, value=top).font = bold
            ws.cell(row=1, column=j).alignment = Alignment(horizontal="center")
        if j == len(groups) or groups[j] != top:
            if j > start:
                ws.merge_cells(start_row=1, start_column=start, end_row=1, end_column=j)
    for row in ws.iter_rows(min_row=3, min_col=4):
        for c in row:
            c.number_format = "0.000"
    for row in ws.iter_rows(min_row=3, max_col=1):
        row[0].font = bold

    residuals_out.rename_axis("date").to_excel(writer, sheet_name="Residuals")
    for row in writer.sheets["Residuals"].iter_rows(min_row=2, min_col=2):
        for c in row:
            c.number_format = "0.0000"

    covariance.rename_axis("ticker").to_excel(writer, sheet_name="Covariance")
    ws = writer.sheets["Covariance"]
    for i, row in enumerate(ws.iter_rows(min_row=2, min_col=2)):
        for j, c in enumerate(row):
            c.number_format = "0.0000"
            if i == j:
                c.font = Font(bold=True)

print(f"saved {output_file.resolve().name}")